# 5 · Amazon's Choice: badge-rate model

Outcome: badged scrapes / scrapes (the badge flickers, and "ever badged" just tracks scrape count). Results are about badges *seen by the scraper*.

In [1]:
import logging, os, warnings
os.environ.setdefault('PYTENSOR_FLAGS', 'cxx=')  # PyTensor's C linker flags break on this macOS toolchain; nutpie compiles via numba anyway
for k, v in {'HF_HUB_VERBOSITY': 'error', 'HF_HUB_DISABLE_PROGRESS_BARS': '1', 'TRANSFORMERS_VERBOSITY': 'error'}.items():
    os.environ.setdefault(k, v)  # quiet model-download chatter
warnings.filterwarnings('ignore', category=UserWarning)  # PyTensor loop-fusion / numba notices
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
logging.getLogger('pymc').setLevel(logging.ERROR)  # after import: pymc sets its own level on import

s = pd.read_csv('../data/amazon_sales_data_cleaned.csv')
d = pd.read_csv('../data/amazon_products_deduped.csv')
d = d.join(s.groupby('Title').agg(sponsored_share=('is_sponsored', 'mean'), in_stock_share=('in_stock', 'mean')), on='Title')
d = d[d['current_price'].notna()].reset_index(drop=True)
d['price_pct'] = d.groupby('category')['current_price'].rank(pct=True)
n, k = d['badge_scrapes'].to_numpy(), d['badge_hits'].to_numpy()
print(f'{len(d)} products, {n.sum()} scrapes, {k.sum()} badged scrapes ({k.sum() / n.sum():.2%}), '
      f'{(k > 0).sum()} products badged at least once')

5830 products, 28248 scrapes, 607 badged scrapes (2.15%), 228 products badged at least once


## Features

All standardized: within-category price percentile, rating, log reviews, log sales, share of scrapes in stock, share sponsored, coupon.

In [2]:
feat = pd.DataFrame({
    'price_pct': d['price_pct'], 'rating': d['rating'].fillna(d['rating'].median()),
    'log_reviews': np.log1p(d['number_of_reviews'].fillna(0)), 'log_sales': np.log1p(d['bought_in_last_month'].fillna(0)),
    'in_stock_share': d['in_stock_share'], 'sponsored_share': d['sponsored_share'], 'has_coupon': d['has_coupon'].astype(float),
})
X = ((feat - feat.mean()) / feat.std()).to_numpy()
cat_idx, cats = pd.factorize(d['category'])
brand_idx, brands = pd.factorize(d['brand'])
feat.describe().round(2)

,price_pct,rating,log_reviews,log_sales,in_stock_share,sponsored_share,has_coupon
count,5830.00,5830.00,5830.00,5830.00,5830.00,5830.00,5830.00
mean,0.50,4.45,6.72,5.72,0.91,0.05,0.03
std,0.29,0.29,2.18,1.86,0.28,0.22,0.16
min,0.00,1.00,0.00,0.00,0.00,0.00,0.00
25%,0.25,4.30,5.31,4.62,1.00,0.00,0.00
50%,0.50,4.50,6.85,5.99,1.00,0.00,0.00
75%,0.75,4.60,8.22,6.91,1.00,0.00,0.00
max,1.00,5.00,13.67,11.51,1.00,1.00,1.00


## Model

$$\text{hits}_i \sim \text{BetaBinomial}(n_i, \rho_i\kappa, (1-\rho_i)\kappa),\quad \text{logit}\,\rho_i = b_0 + a^{cat} + a^{brand} + x_i^\top\beta$$

Beta-Binomial absorbs product-level noise; category and brand partially pooled.

In [3]:
def build(X, cat_idx, brand_idx, n, k, brand_level=True):
    with pm.Model() as model:
        b0 = pm.Normal('b0', np.log(0.02 / 0.98), 1)
        beta = pm.Normal('beta', 0, 0.5, shape=X.shape[1])
        s_cat = pm.HalfNormal('s_cat', 1)
        a_cat = pm.Deterministic('a_cat', s_cat * pm.Normal('z_cat', 0, 1, shape=len(cats)))
        eta = b0 + a_cat[cat_idx] + pm.math.dot(X, beta)
        if brand_level:
            s_brand = pm.HalfNormal('s_brand', 1)
            a_brand = pm.Deterministic('a_brand', s_brand * pm.Normal('z_brand', 0, 1, shape=len(brands)))
            eta = eta + a_brand[brand_idx]
        rho = pm.Deterministic('rho', pm.math.sigmoid(eta))
        kappa = pm.Gamma('kappa', 2, 0.1)
        pm.BetaBinomial('hits', n=n, alpha=rho * kappa, beta=(1 - rho) * kappa, observed=k)
    return model

model = build(X, cat_idx, brand_idx, n, k)
with model:
    idata = pm.sample(1000, tune=2000, chains=4, nuts_sampler='nutpie', random_seed=0, progressbar=False)
    pm.compute_log_likelihood(idata)
diag = az.summary(idata, var_names=['~rho', '~a_'], filter_vars='like')
print(f"max R-hat {diag['r_hat'].max():.3f}, min ESS {diag['ess_bulk'].min():.0f}, "
      f"divergences {int(idata.sample_stats['diverging'].sum())}")
az.summary(idata, var_names=['b0', 's_cat', 's_brand', 'kappa']).round(3)

max R-hat 1.009, min ESS 467, divergences 0


,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
b0,-4.002,0.108,-4.2,-3.8,1889,2240,1.00,0.0025,0.0017
s_cat,0.126,0.092,0.011,0.3,1334,1502,1.00,0.0024,0.0018
s_brand,0.31,0.158,0.05,0.56,467,539,1.00,0.0072,0.0029
kappa,14.81,2.8,11,20,3049,2803,1.00,0.048,0.05


## Feature effects (odds multiplier per SD)

In [4]:
post = idata.posterior.to_dataset().stack(s=('chain', 'draw'))
draws = lambda v: post[v].to_numpy()
b = draws('beta')
effects = pd.DataFrame({'feature': feat.columns, 'odds x per SD': np.exp(np.median(b, 1)),
                        'lo 90%': np.exp(np.quantile(b, .05, 1)), 'hi 90%': np.exp(np.quantile(b, .95, 1)),
                        'P(effect > 0)': (b > 0).mean(1)})
effects.round(3)

,feature,odds x per SD,lo 90%,hi 90%,P(effect > 0)
0,price_pct,0.903,0.800,1.013,0.074
1,rating,1.356,1.206,1.530,1.000
2,log_reviews,1.116,0.978,1.275,0.912
3,log_sales,1.252,1.101,1.428,0.998
4,in_stock_share,1.259,1.088,1.500,0.995
5,sponsored_share,1.226,1.150,1.313,1.000
6,has_coupon,1.012,0.926,1.093,0.597


Rating, sales, stock and sponsorship each raise the odds ~1.2-1.4x; price leans cheaper-is-better; reviews and coupons add nothing.

## Categories and brands

In [5]:
def group_table(var, names, counts):
    a = draws(var)
    return pd.DataFrame({'group': names, 'products': counts, 'odds x': np.exp(np.median(a, 1)),
                         'lo 90%': np.exp(np.quantile(a, .05, 1)), 'hi 90%': np.exp(np.quantile(a, .95, 1))})

cat_tab = group_table('a_cat', cats, pd.Series(cats).map(d['category'].value_counts()).to_numpy())
cat_tab.sort_values('odds x', ascending=False).round(2)

,group,products,odds x,lo 90%,hi 90%
0,Storage & Memory Cards,298,1.04,0.92,1.45
4,Computers & Laptops,799,1.03,0.90,1.35
15,Keyboards & Mice,208,1.02,0.87,1.39
13,TV & Home Entertainment,133,1.02,0.87,1.37
14,Cables & Charging,328,1.02,0.87,1.32
5,Networking,249,1.02,0.86,1.35
3,Phones & Tablets,91,1.01,0.83,1.34
12,Cameras & Photography,538,1.01,0.85,1.27
19,Optics,38,1.00,0.79,1.28
7,Monitors & Displays,236,1.00,0.79,1.25


Categories: all ~1.0.

In [6]:
brand_tab = group_table('a_brand', brands, pd.Series(brands).map(d['brand'].value_counts()).to_numpy())
print('Amazon brands:')
display(brand_tab[brand_tab['group'].isin(['Amazon', 'Amazon Basics'])].round(2))
print('Brands whose 90% interval excludes 1 (or top/bottom 8 with 10+ products):')
clear = brand_tab[(brand_tab['lo 90%'] > 1) | (brand_tab['hi 90%'] < 1)]
big = brand_tab[brand_tab['products'] >= 10].sort_values('odds x')
pd.concat([clear, big.head(8), big.tail(8)]).drop_duplicates('group').sort_values('odds x').round(2)

Amazon brands:


,group,products,odds x,lo 90%,hi 90%
2,Amazon Basics,159,0.94,0.55,1.36


Brands whose 90% interval excludes 1 (or top/bottom 8 with 10+ products):


,group,products,odds x,lo 90%,hi 90%
37,Asus,141,0.81,0.39,1.18
181,Peak,14,0.82,0.37,1.21
45,Kodak,42,0.83,0.39,1.21
180,Ubeesize,25,0.86,0.43,1.25
6,Sandisk,70,0.87,0.44,1.24
7,Amd,23,0.90,0.46,1.34
18,Dell,94,0.90,0.48,1.33
10,Acer,62,0.91,0.49,1.34
184,Godox,38,1.13,0.78,2.14
110,Shure,26,1.13,0.77,2.30


Brands: none clearly differ; Amazon Basics 0.94 (0.55-1.36), no private-label advantage.

## Does brand add anything? (LOO)

In [7]:
with build(X, cat_idx, brand_idx, n, k, brand_level=False):
    idata_nb = pm.sample(1000, tune=2000, chains=4, nuts_sampler='nutpie', random_seed=0, progressbar=False)
    pm.compute_log_likelihood(idata_nb)
az.compare({'category + brand': idata, 'category only': idata_nb}).round(2)

,rank,elpd_diff,dse,p_worse,diag_diff,diag_elpd,p,elpd,se,weight
category only,0,0.0,0.0,NaN,,,13.2,-1000.0,53.0,0.55
category + brand,1,-0.0,1.8,0.54,|elpd_diff| < 4,,35.7,-1000.0,53.0,0.45


No: difference ~0.

## Posterior predictive check: ever-badged vs scrape count

In [8]:
with model:
    ppc = pm.sample_posterior_predictive(idata, random_seed=0, progressbar=False)
sim = ppc.posterior_predictive.to_dataset()['hits'].stack(s=('chain', 'draw')).to_numpy() > 0
bucket = pd.cut(n, [0, 1, 2, 4, 19, 49, 10_000], labels=['1', '2', '3-4', '5-19', '20-49', '50+'])
pd.DataFrame({'products': pd.Series(bucket).value_counts().sort_index(),
              'observed ever-badged': pd.Series(k > 0).groupby(bucket).mean(),
              'model (median)': pd.DataFrame(sim).groupby(bucket).mean().median(axis=1),
              'model lo 90%': pd.DataFrame(sim).groupby(bucket).mean().quantile(.05, axis=1),
              'model hi 90%': pd.DataFrame(sim).groupby(bucket).mean().quantile(.95, axis=1)}).round(3)

,products,observed ever-badged,model (median),model lo 90%,model hi 90%
1,5053,0.019,0.022,0.018,0.027
2,424,0.050,0.038,0.024,0.057
3-4,46,0.152,0.087,0.022,0.174
5-19,80,0.262,0.250,0.175,0.338
20-49,60,0.250,0.317,0.217,0.400
50+,167,0.401,0.395,0.323,0.467


Reproduced in every bucket: heavily scraped products aren't special, just seen more.